## Author: Chan Jin Wei
---

In [1]:
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from classes.data_cleaner import DataCleaner
from classes.dataframe_saver import DataFrameSaver
import subprocess

In [2]:
# Delete old HDFS version
subprocess.run(['hdfs', 'dfs', '-rm', 'cleaned_articles/*'])

Deleted cleaned_articles/_SUCCESS
Deleted cleaned_articles/cleaned_articles.csv


CompletedProcess(args=['hdfs', 'dfs', '-rm', 'cleaned_articles/*'], returncode=0)

In [3]:
spark = SparkSession.builder.appName("dataCleaner").master("local").getOrCreate()

24/12/22 21:16:36 WARN Utils: Your hostname, LAPTOP-PFPL3CLD. resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
24/12/22 21:16:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/12/22 21:16:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/12/22 21:16:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/12/22 21:16:37 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [4]:
input_path = "articles/articles.csv"
df = DataCleaner.csvToDataFrame(input_path, spark)

In [5]:
# Clean the data
df = DataCleaner.remove_numbers(df, column="article")
df = DataCleaner.convert_to_lowercase(df, column="article")

df = DataCleaner.remove_standalone_characters(df, column="article")
df = DataCleaner.remove_words_in_parentheses(df, column="article")

# Remove delimiters
delimiters_to_remove = r',.|;$&?!():@\'\"\[\]\<\>/{/}/‘/’/“/”'
df = DataCleaner.remove_delimiter(df, column="article", delimiters=delimiters_to_remove)

# Conditional delimiter removal
delimiters_to_remove = "-'"
df = DataCleaner.smart_remove_delimiter(df, column="article", delimiters=delimiters_to_remove)
df = DataCleaner.remove_hyphen_from_prefix(df, column="article")

# Normalize spaces
df = DataCleaner.normalize_spaces(df, column="article")

# Split 'article' column into words and create a new DataFrame
rdd = df.rdd.flatMap(lambda line: line['article'].split(" ")) 
df_split = rdd.map(lambda word: (word,)).toDF(["words"]) 

Removing delimiters: ,.|;$&?!():@\'\"\[\]\<\>/{/}/‘/’/“/” from column: article


In [6]:
output_dir = "/home/student/de-assgt/content"
file_name = "cleaned_articles.csv"

DataFrameSaver.save_to_csv(df_split, output_dir, file_name)

print(f"Cleaned article content saved as {file_name} in {output_dir}")
df_split.show()

Cleaned article content saved as cleaned_articles.csv in /home/student/de-assgt/content
+----------+
|     words|
+----------+
|   banting|
|     polis|
|    diraja|
|  malaysia|
|   menahan|
|      enam|
|    lelaki|
|     warga|
|  tempatan|
|dipercayai|
|  terlibat|
|     dalam|
|  kejadian|
|   rusuhan|
|        di|
|  jenjarom|
|        di|
|      sini|
|      pada|
|    khamis|
+----------+
only showing top 20 rows



In [7]:
hdfs_path = "cleaned_articles"
df_split.coalesce(1).write.csv(hdfs_path, header=True, mode="overwrite")
print(f"Article content saved to HDFS at: {hdfs_path}")

Article content saved to HDFS at: cleaned_articles


In [8]:
df_split.count()

192

In [9]:
# Rename part-00000 file
subprocess.run(['hdfs', 'dfs', '-mv', 'cleaned_articles/part-00000*', 'cleaned_articles/cleaned_articles.csv'])

CompletedProcess(args=['hdfs', 'dfs', '-mv', 'cleaned_articles/part-00000*', 'cleaned_articles/cleaned_articles.csv'], returncode=0)

In [10]:
spark.stop()